# Sofia ML Regression 2026 — Robust Two-Regime Ensemble

**Author:** Santosh Bogati


---
## Section 1 — Import Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
plt.rcParams['figure.facecolor'] = '#F8F9FA'
plt.rcParams['axes.facecolor']   = '#FFFFFF'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings('ignore')


---
## Section 2 — Load Data




In [ ]:
# Update paths to your local dataset copies
# Download: https://www.kaggle.com/competitions/sofia-ml-regression-2026-winter/data
train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')

features = [f'x{i}' for i in range(15)]
y = train['target'].values

print(f'Train: {train.shape[0]:,} rows × {train.shape[1]} cols')
print(f'Test : {test.shape[0]:,} rows × {test.shape[1]} cols')
train.head()


---
## Section 3 — Exploratory Data Analysis (EDA)




### 3A — Basic Statistics

In [ ]:
print('=== TARGET STATISTICS ===')
print(f'  Min    : {y.min():.2f}')
print(f'  Mean   : {y.mean():.2f}')
print(f'  Median : {np.median(y):.2f}')
print(f'  Max    : {y.max():.2f}')
print(f'  Std    : {y.std():.2f}')
print()
print('Note: Mean (81) >> Median (7) — strong right skew from extreme values')
train[features].describe().round(2)


### 3B — Missing Values

In [ ]:
missing = train[features].isnull().sum()
missing_pct = (missing / len(train) * 100).round(2)

pd.DataFrame({'Missing Count': missing, 'Missing %': missing_pct})


In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
bars = ax.bar(missing_pct.index, missing_pct.values, color='#1C7293', width=0.6, edgecolor='none')
ax.set_title('Missing Values per Feature (%)', fontsize=14, fontweight='bold', pad=10)
ax.set_ylabel('Missing %', fontsize=11)
ax.set_ylim(0, 4)
for bar, val in zip(bars, missing_pct.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.05,
            f'{val:.1f}%', ha='center', fontsize=9, color='#2C3E50')
ax.axhline(2.0, color='#E74C3C', linestyle='--', linewidth=1, alpha=0.5, label='2% reference')
ax.legend(fontsize=10)
plt.tight_layout()
plt.show()


### 3C — Target Distribution & Outlier Analysis

This is the **most important discovery** in the entire analysis.  
We use the **IQR (Interquartile Range)** method to identify outliers:
.

In [ ]:
q1  = np.percentile(y, 25)
q3  = np.percentile(y, 75)
iqr = q3 - q1
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr

mask_normal  = (y >= lower) & (y <= upper)
mask_extreme = ~mask_normal

print(f'Normal rows  : {mask_normal.sum()} ({mask_normal.mean()*100:.1f}%)')
print(f'Extreme rows : {mask_extreme.sum()} ({mask_extreme.mean()*100:.1f}%)')
print()

total_var   = np.sum((y - y.mean())**2)
extreme_var = np.sum((y[mask_extreme] - y.mean())**2)
print(f'Variance explained by extreme rows: {extreme_var/total_var*100:.1f}%')
print('7% of rows, 94.6% of variance — any model that misses these scores near R²=0')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

imp_tmp = SimpleImputer(strategy='median')
X_tmp   = pd.DataFrame(imp_tmp.fit_transform(train[features]), columns=features)

axes[0].hist(y[mask_normal],  bins=80, color='#1C7293', edgecolor='none', alpha=0.85, label=f'Normal ({mask_normal.sum()})')
axes[0].hist(y[mask_extreme], bins=30, color='#E74C3C', edgecolor='none', alpha=0.85, label=f'Extreme ({mask_extreme.sum()})')
axes[0].set_title('Target Distribution — Two Regimes', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Target Value', fontsize=11)
axes[0].set_ylabel('Count', fontsize=11)
axes[0].legend(fontsize=10)

axes[1].scatter(X_tmp['x1'][mask_normal],  y[mask_normal],  s=4,  alpha=0.25, color='#1C7293', label='Normal')
axes[1].scatter(X_tmp['x1'][mask_extreme], y[mask_extreme], s=20, alpha=0.8,  color='#E74C3C', label='Extreme')
axes[1].set_title('Target vs x1 — Two-Pattern Structure', fontsize=14, fontweight='bold')
axes[1].set_xlabel('x1', fontsize=11)
axes[1].set_ylabel('target', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

x1_sq_ext = X_tmp['x1'][mask_extreme]**2
ax.scatter(x1_sq_ext, y[mask_extreme], s=25, alpha=0.7, color='#E74C3C', label='Extreme rows (actual)')

xline = np.linspace(0, 1100, 300)
ax.plot(xline, 5*xline, color='#F39C12', linewidth=2.5, label='y = 5 × x1²')

ax.set_title('Extreme rows: target ≈ 5 × x1²', fontsize=14, fontweight='bold')
ax.set_xlabel('x1²', fontsize=11)
ax.set_ylabel('target', fontsize=11)
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

pred_formula = 5.0 * x1_sq_ext
print(f'R² of 5×x1² on extreme rows: {r2_score(y[mask_extreme], pred_formula):.4f}')


### 3D — Feature Correlations with Target

Since extreme rows dominate the raw data, we inspect correlations on **clean (normal) rows only**.  


In [ ]:
corrs_normal = pd.Series({
    f: X_tmp.loc[mask_normal, f].corr(pd.Series(y[mask_normal]))
    for f in features
}).sort_values()

print('Feature correlations with target (normal rows only):')
print(corrs_normal.round(4).to_string())


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
colors = ['#E74C3C' if v < 0 else '#27AE60' for v in corrs_normal]
bars = ax.barh(corrs_normal.index, corrs_normal.values, color=colors, edgecolor='none', height=0.6)
ax.axvline(0, color='#2C3E50', linewidth=1.2)
ax.set_title('Feature Correlations with Target (Normal Rows Only)', fontsize=14, fontweight='bold', pad=10)
ax.set_xlabel('Pearson Correlation', fontsize=11)
for bar, val in zip(bars, corrs_normal.values):
    offset = 0.01 if val >= 0 else -0.01
    ha = 'left' if val >= 0 else 'right'
    ax.text(val + offset, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', ha=ha, fontsize=9)
plt.tight_layout()
plt.show()


---
## Section 4 — Preprocessing

### Median Imputation for extreme median handling


In [ ]:
# Median is robust to the extreme values — mean imputation would be skewed
imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(train[features]), columns=features)
X_test = pd.DataFrame(imp.transform(test[features]),      columns=features)

print(f'Missing after imputation — train: {X.isnull().sum().sum()}, test: {X_test.isnull().sum().sum()}')


---
## Section 5 — Feature Engineering (Most Important part here)

We added the **squared version of every feature** (x0², x1², ... x14²) bringing the total from 15 to 30 features.

**Why squared features?**
- The relationship between x1, specifically, and target is non-linear (parabolic for extreme rows).
- Adding x² allows a linear model to fit curved patterns.
- The linear model can now learn: `target ≈ a·x4 + b·x2 + c·x1²` — capturing both regimes.

The **same function** is applied to both train and test to ensure consistency.

In [ ]:
def build_features(df):
    out = df.copy()
    for f in features:
        out[f'{f}_sq'] = df[f] ** 2
    return out

X_eng      = build_features(X)
X_test_eng = build_features(X_test)

print(f'Features: {X.shape[1]} → {X_eng.shape[1]} (added squared terms)')


---
## Section 6 — Model Training

### Key Decision: Train on Normal Rows Only



In [ ]:
q1, q3 = np.percentile(y, 25), np.percentile(y, 75)
iqr     = q3 - q1
mask_normal = (y >= q1 - 1.5*iqr) & (y <= q3 + 1.5*iqr)

print(f'Training on {mask_normal.sum()} normal rows — {(~mask_normal).sum()} extreme rows handled by formula')


In [ ]:
sc_final = StandardScaler()
m_final  = LinearRegression()
m_final.fit(sc_final.fit_transform(X_eng[mask_normal]), y[mask_normal])

train_r2 = r2_score(y[mask_normal], m_final.predict(sc_final.transform(X_eng[mask_normal])))
print(f'R² on normal rows: {train_r2:.4f}')

coef_df = pd.DataFrame({
    'Feature'    : X_eng.columns,
    'Coefficient': m_final.coef_
}).sort_values('Coefficient', key=abs, ascending=False)

print()
print('Top 10 features by |coefficient|:')
print(coef_df.head(10).to_string(index=False))


---
## Section 7 — Cross Validation



In [ ]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(y))
fold_r2s  = []

for fold, (tr, val) in enumerate(kf.split(X_eng)):
    mask_tr_norm = mask_normal[tr]

    sc = StandardScaler()
    m  = LinearRegression()
    m.fit(sc.fit_transform(X_eng.iloc[tr][mask_tr_norm]), y[tr][mask_tr_norm])

    p_linear = m.predict(sc.transform(X_eng.iloc[val]))
    p_formula = 5.0 * X.iloc[val]['x1'].values**2

    blend = 0.9 * p_linear + 0.1 * p_formula
    oof_preds[val] = blend

    fr2 = r2_score(y[val], blend)
    fold_r2s.append(fr2)
    print(f'  Fold {fold+1}: R² = {fr2:.4f}')

print()
print(f'Mean CV R²  : {np.mean(fold_r2s):.4f}')
print(f'CV RMSE     : {np.sqrt(mean_squared_error(y, oof_preds)):.4f}')


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y[mask_normal],  oof_preds[mask_normal],  s=4,  alpha=0.3, color='#1C7293', label='Normal')
axes[0].scatter(y[~mask_normal], oof_preds[~mask_normal], s=20, alpha=0.7, color='#E74C3C', label='Extreme')
lims = [min(y.min(), oof_preds.min()), max(y.max(), oof_preds.max())]
axes[0].plot(lims, lims, 'k--', lw=1.5, label='Perfect fit')
axes[0].set_title('Actual vs Predicted (OOF)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Actual', fontsize=11)
axes[0].set_ylabel('Predicted', fontsize=11)
axes[0].legend(fontsize=10)

resid = y - oof_preds
axes[1].hist(resid[mask_normal],  bins=60, color='#1C7293', alpha=0.8, edgecolor='none', label='Normal')
axes[1].hist(resid[~mask_normal], bins=20, color='#E74C3C', alpha=0.8, edgecolor='none', label='Extreme')
axes[1].axvline(0, color='black', lw=1.5)
axes[1].set_title('Residuals', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Residual', fontsize=11)
axes[1].set_ylabel('Count', fontsize=11)
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()


## Section 8 — Train Final Model & Generate Submission


In [ ]:
import pandas as pd
import numpy as np
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

train = pd.read_csv('data/train.csv')
test  = pd.read_csv('data/test.csv')
features = [f'x{i}' for i in range(15)]
y = train['target'].values

imp    = SimpleImputer(strategy='median')
X      = pd.DataFrame(imp.fit_transform(train[features]), columns=features)
X_test = pd.DataFrame(imp.transform(test[features]),      columns=features)

def build_features(df):
    out = df.copy()
    for f in features:
        out[f'{f}_sq'] = df[f] ** 2
    return out

X_eng      = build_features(X)
X_test_eng = build_features(X_test)

q1, q3 = np.percentile(y, 25), np.percentile(y, 75)
mask_normal = (y >= q1 - 1.5*(q3-q1)) & (y <= q3 + 1.5*(q3-q1))

sc = StandardScaler()
m  = LinearRegression()
m.fit(sc.fit_transform(X_eng[mask_normal]), y[mask_normal])

# 90/10 blend — weight found via cross-validation sweep
p_linear  = m.predict(sc.transform(X_test_eng))
p_formula = 5.0 * X_test['x1'].values**2
final_preds = 0.9 * p_linear + 0.1 * p_formula

submission = pd.DataFrame({'Id': test['Id'], 'target': final_preds})
submission.to_csv('submission.csv', index=False)

print(f'Saved submission.csv — {len(submission)} rows')
submission.head(10)


## Section 9 — Summary


### What made this solution work

The key was treating the dataset as two structurally different problems rather than one:

- **Normal rows (~93%)** — a linear relationship captured well by LinearRegression on 30 features (15 raw + 15 squared)
- **Extreme rows (~7%)** — a parabolic pattern `target ≈ 5 × x1²` derived from data inspection, not learned by the model

Blending both at 90/10 gave the best cross-validated RMSE (~0.0819) and ranked **#1 on the private leaderboard**.

The public leaderboard score was low because the public test set contained no extreme rows — the formula had nothing to act on. The private set did.

**Takeaway:** EDA that reveals data structure beats model complexity every time.
